# Posit Connect Inventory Scan

Discovers deployed content and stores its package inventory in PostgreSQL.

Configuration comes from environment variables set in this content item's **Vars** panel.

In [ ]:
from clients.connect_client import ConnectClient
from config.logging_config import configure_logging
from config.settings import get_settings
from database.connection import get_database
from services.inventory_service import InventoryService

configure_logging(level='INFO', log_dir=None, force=True)
settings = get_settings()
print('Connect:', settings.connect_server_url)

## Connect and prepare the schema

In [ ]:
from database.migrations import head_revision

database = get_database(settings)
database.verify_connection()

# The database is only reachable from inside the VNet, so migrations run
# here rather than from a workstation. Safe on every run.
database.ensure_schema()
database.verify_schema()
print('Schema ready (revision ' + head_revision() + ')')

## Run the scan

In [ ]:
def on_discovered(count):
    print('Found', count, 'applications. Collecting packages...')


with ConnectClient(settings) as client:
    info = client.verify_connection()
    print('Connected as', info['username'], '(role=' + str(info['user_role']) + ')')

    service = InventoryService(client, database, settings)
    result = service.run(on_discovered=on_discovered)

print()
print('Applications stored:', result.applications_stored)
print('Packages stored    :', result.packages_stored)

## Results

In [ ]:
if result.failures:
    print(len(result.failures), 'application(s) could not be inventoried:')
    for guid, message in result.failures[:10]:
        print(' -', guid, ':', message[:120])
else:
    print('No failures.')

In [ ]:
from repositories.application_repository import ApplicationRepository
from repositories.package_repository import PackageRepository

with database.session() as session:
    apps = ApplicationRepository(session)
    packages = PackageRepository(session)
    print('Stored in PostgreSQL')
    print('  applications        :', apps.count())
    print('  packages            :', packages.count())
    print('  distinct owners     :', apps.distinct_owner_count())
    print('  unique pkg versions :', packages.distinct_package_versions())
    print('  by runtime          :', packages.package_type_breakdown())

database.dispose()

## CSV export for the OIS scanning team

In [ ]:
import base64
from IPython.display import HTML, display

from services.export_service import ExportService

export = ExportService(database)
csv_text = export.to_string()
filename = export.suggested_filename()
print(csv_text.count(chr(13) + chr(10)) - 1, 'rows')

# Embedded so the link works from the rendered report, which is static HTML.
payload = base64.b64encode(csv_text.encode('utf-8-sig')).decode('ascii')
display(HTML(
    '<a download="' + filename + '" '
    'href="data:text/csv;base64,' + payload + '" '
    'style="display:inline-block;padding:10px 18px;background:#447099;'
    'color:#fff;border-radius:4px;text-decoration:none;font-weight:600">'
    'Download ' + filename + '</a>'
))

## Zip download for the DevSecOps upload

In [ ]:
zip_name = export.suggested_filename(extension='zip')
zip_bytes = export.to_zip_bytes(arcname=filename)

zip_payload = base64.b64encode(zip_bytes).decode('ascii')
display(HTML(
    '<a download="' + zip_name + '" '
    'href="data:application/zip;base64,' + zip_payload + '" '
    'style="display:inline-block;padding:10px 18px;background:#447099;'
    'color:#fff;border-radius:4px;text-decoration:none;font-weight:600">'
    'Download ' + zip_name + '</a>'
))

## CycloneDX SBOMs for Wiz

One SBOM per application, as OIS specified. Findings come back per
SBOM, which is what maps a vulnerability to an app and its owner.

In [ ]:
sbom_bytes, sbom_count = export.to_sbom_zip_bytes()
sbom_name = export.suggested_filename(prefix='connect-sboms', extension='zip')
print(sbom_count, 'SBOMs')

sbom_payload = base64.b64encode(sbom_bytes).decode('ascii')
display(HTML(
    '<a download="' + sbom_name + '" '
    'href="data:application/zip;base64,' + sbom_payload + '" '
    'style="display:inline-block;padding:10px 18px;background:#447099;'
    'color:#fff;border-radius:4px;text-decoration:none;font-weight:600">'
    'Download ' + sbom_name + '</a>'
))